In [ ]:
# CELL 1: IMPORTS
import datetime, warnings, contextlib, io, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from arch.unitroot import ADF
except ImportError:
    !pip install arch -q
    from arch.unitroot import ADF

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 7)

In [ ]:
# CELL 2: CONFIG + DATA LOADING
DATA_SOURCE = "fred"               # 'fred' (free, no API key) or 'csv'
FREQ = "M"                         # 'M' monthly | 'Q' quarterly
START_DATE = "1995-01-01"
END_DATE = datetime.date.today().strftime("%Y-%m-%d")
TRAINING_DATA_MODE = "full"        # 'full' | 'recent_years' | 'custom_range'
TRAIN_YEARS = 25                   # used if 'recent_years'
EXCLUDE_COVID = True               # drop 2020-03 to 2021-06 rows from modelling
LAG_CRITERION = "bic"              # few obs -> BIC is safer
SCALE = 100                        # logdiff x100 = % change

MAX_LAGS       = {"M": 12, "Q": 4}
FORECAST_STEPS = {"M": 12, "Q": 4}
PER_YEAR       = {"M": 12, "Q": 4}
BAR    = {"M": pd.offsets.MonthEnd(), "Q": pd.offsets.QuarterEnd()}
BAR_TD = {"M": pd.Timedelta("28D"),   "Q": pd.Timedelta("85D")}

# transform: 'logdiff' -> index/level series (CPI, IP, payrolls, M2, retail sales)
#            'diff'    -> rates and % series (unemployment, Fed funds, yields, spreads)
INDICATORS = [
    {"alias": "CPI",      "fred_id": "CPIAUCSL", "csv_file_path": r"D:\macro\cpi.csv",      "csv_column": "CPI",      "transform": "logdiff"},
    {"alias": "UNRATE",   "fred_id": "UNRATE",   "csv_file_path": r"D:\macro\unrate.csv",   "csv_column": "UNRATE",   "transform": "diff"},
    {"alias": "INDPRO",   "fred_id": "INDPRO",   "csv_file_path": r"D:\macro\indpro.csv",   "csv_column": "INDPRO",   "transform": "logdiff"},
    {"alias": "FEDFUNDS", "fred_id": "FEDFUNDS", "csv_file_path": r"D:\macro\fedfunds.csv", "csv_column": "FEDFUNDS", "transform": "diff"},
]
DATE_NAMES = ["date", "datetime", "time", "timestamp", "observation_date"]


def fred_series(inst):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={inst['fred_id']}"
    df = pd.read_csv(url)
    return pd.Series(pd.to_numeric(df.iloc[:, 1], errors="coerce").values,
                     index=pd.to_datetime(df.iloc[:, 0]), name=inst["alias"]).dropna()


def read_csv_series(inst):
    df = pd.read_csv(inst["csv_file_path"])
    df.columns = df.columns.str.strip()
    date_col = next((c for c in df.columns if c.lower() in DATE_NAMES), df.columns[0])
    want = inst["csv_column"]
    if want not in df.columns:
        raise KeyError(f"'{want}' not in {list(df.columns)}")
    s = pd.Series(pd.to_numeric(df[want], errors="coerce").values,
                  index=pd.to_datetime(df[date_col], format="mixed"), name=inst["alias"])
    return s.dropna().sort_index()


def load_macro():
    loader = fred_series if DATA_SOURCE == "fred" else read_csv_series
    series = []
    for inst in INDICATORS:
        s = loader(inst)
        s = s[s.index >= START_DATE]
        if s.index.to_series().diff().median() > BAR_TD[FREQ] * 1.5:
            raise ValueError(f"{inst['alias']} is coarser than FREQ='{FREQ}' (e.g. GDP needs FREQ='Q')")
        series.append(s)
    df = pd.concat(series, axis=1).sort_index().resample(BAR[FREQ]).last()
    print("Last available date per series (shortest one limits the sample):")
    print(df.apply(lambda s: s.last_valid_index()).to_string())
    df = df.dropna()
    if TRAINING_DATA_MODE == "recent_years":
        df = df.loc[df.index[-1] - pd.DateOffset(years=TRAIN_YEARS):]
    elif TRAINING_DATA_MODE == "custom_range":
        df = df.loc[START_DATE:END_DATE]
    print(f"\n[{FREQ}] {len(df)} obs | {df.index[0].date()} -> {df.index[-1].date()}")
    if len(df) < 100:
        print("WARNING: < 100 obs - use fewer variables / lags.")
    return df


data = load_macro()
data.tail()

In [ ]:
# CELL 3: FIGURE 1 - TIME PLOTS (LEVELS)
n = len(data.columns)
fig, axs = plt.subplots(n, 1, sharex=True, figsize=(12, 3 * n))
for ax, col in zip(np.atleast_1d(axs), data.columns):
    data[col].plot(ax=ax, linewidth=1, xlabel="Date", ylabel=col, title=col)
fig.tight_layout()
plt.show()

In [ ]:
# CELL 4: FIGURE 2 - ADF TEST ON LEVELS (5% level, trend='c', BIC)
def adf_table(df):
    rows = {}
    for c in df.columns:
        a = ADF(df[c].dropna(), trend="c", method="bic")
        rows[c] = {"ADF Test Statistic": a.stat, "5% Critical Value": a.critical_values["5%"],
                   "p-value": a.pvalue, "Stationary": a.stat < a.critical_values["5%"]}
    return pd.DataFrame(rows).T.round(4)

print("--- ADF on LEVELS (Stationary=False -> unit root) ---")
adf_table(data)

In [ ]:
# CELL 5: FIGURES 3 & 4 - TRANSFORM (logdiff or diff) + ADF
def make_stationary(df):
    out = pd.DataFrame(index=df.index)
    for i in INDICATORS:
        a = i["alias"]
        out[a] = np.log(df[a]).diff() * SCALE if i["transform"] == "logdiff" else df[a].diff()
    return out.dropna()

diff_data = make_stationary(data)

if EXCLUDE_COVID:
    covid_rows = diff_data.loc["2020-03":"2021-06"].index
    diff_data = diff_data.drop(covid_rows)
    print(f"Excluded {len(covid_rows)} COVID rows")

fig, axs = plt.subplots(n, 1, sharex=True, figsize=(12, 3 * n))
for ax, col in zip(np.atleast_1d(axs), diff_data.columns):
    diff_data[col].plot(ax=ax, linewidth=1, xlabel="Date", ylabel=f"{col} change", title=f"{col} (stationary form)")
fig.tight_layout()
plt.show()

print("--- ADF on TRANSFORMED data ---")
adf_table(diff_data)

In [ ]:
# CELL 6: GRANGER CAUSALITY (all ordered pairs)
def granger(df, cause, effect, maxlag=4):
    with contextlib.redirect_stdout(io.StringIO()):
        r = grangercausalitytests(df[[effect, cause]], maxlag=maxlag)
    return pd.DataFrame([{
        "Lag": l,
        "F-Stat": round(r[l][0]["ssr_ftest"][0], 4),
        "p-value": round(r[l][0]["ssr_ftest"][1], 4),
        "Significant (p<0.05)": r[l][0]["ssr_ftest"][1] < 0.05} for l in range(1, maxlag + 1)])

for a, b in itertools.permutations(diff_data.columns, 2):
    print(f"\n--- Does {a} Granger-cause {b}? ---")
    print(granger(diff_data, cause=a, effect=b).to_string(index=False))

In [ ]:
# CELL 7: FIGURE 5 - LAG SELECTION
model = VAR(diff_data)
k = diff_data.shape[1]
max_l = min(MAX_LAGS[FREQ], (len(diff_data) - 1) // (k + 1))
sel = model.select_order(maxlags=max_l, trend="c")
print(sel.summary())

optimal_lag = max(1, sel.selected_orders[LAG_CRITERION])
print(f"\nSelected lag ({LAG_CRITERION.upper()}): {optimal_lag}")

In [ ]:
# CELL 8: FIGURE 6 - FIT VAR(p) WITH INTERCEPT
var_results = model.fit(optimal_lag, trend="c", method="ols")
print(var_results.summary())
lag_order = var_results.k_ar
print("Lag order:", lag_order)

In [ ]:
# CELL 9: DIAGNOSTICS - STABILITY + RESIDUAL WHITENESS
is_stable = var_results.is_stable(verbose=True)
print(f"\nStable / ergodic: {is_stable}")

# one year of residual lags (12 monthly / 4 quarterly) beyond p
wh = var_results.test_whiteness(nlags=var_results.k_ar + PER_YEAR[FREQ])
print(f"Residual whiteness p-value: {round(wh.pvalue, 4)}")

var_results.plot_acorr()
plt.tight_layout()
plt.show()

In [ ]:
# CELL 10: IRF + FEVD (one year horizon)
irf = var_results.irf(periods=PER_YEAR[FREQ])
irf.plot(orth=True)
plt.suptitle("Orthogonalized Impulse Response Functions", y=1.02)
plt.show()

var_results.fevd(periods=PER_YEAR[FREQ]).plot()
plt.show()

In [ ]:
# CELL 11: FIGURE 7 - FORECAST OF TRANSFORMED SERIES
steps = FORECAST_STEPS[FREQ]
var_results.plot_forecast(steps=steps, alpha=0.05, plot_stderr=True)
plt.show()

In [ ]:
# CELL 12: FIGURE 8 - FORECAST OF LEVELS (each series rebuilt by its own transform)
pred = var_results.forecast(y=diff_data.values[-lag_order:], steps=steps)
idx = pd.date_range(start=data.index[-1] + BAR[FREQ], periods=steps, freq=BAR[FREQ])
df_forecast = pd.DataFrame(pred, index=idx, columns=[f"{c}_chg" for c in data.columns])

for i in INDICATORS:
    a = i["alias"]
    last = data[a].iloc[-1]
    cum = df_forecast[f"{a}_chg"].cumsum()
    if i["transform"] == "logdiff":
        df_forecast[f"{a}Forecast"] = np.exp(np.log(last) + cum / SCALE)
    else:
        df_forecast[f"{a}Forecast"] = last + cum

print(df_forecast[[f"{c}Forecast" for c in data.columns]].round(3))

hist = PER_YEAR[FREQ] * 10     # show last 10 years
for col in data.columns:
    plt.figure(figsize=(16, 5))
    data[col].tail(hist).plot(legend=True, label=f"Historical {col}")
    df_forecast[f"{col}Forecast"].plot(legend=True, label=f"Forecast {col}")
    plt.axvline(x=idx[0], color="b", alpha=0.5, linestyle="--")
    plt.title(f"VAR Level Forecast: {col}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# CELL 13: YoY % FORECAST (how inflation / growth is normally read)
py = PER_YEAR[FREQ]
for i in INDICATORS:
    if i["transform"] == "logdiff":
        a = i["alias"]
        path = pd.concat([data[a], df_forecast[f"{a}Forecast"].rename(a)])
        yoy = (path.pct_change(py) * 100).dropna()
        print(f"\n{a} YoY % (forecast part):")
        print(yoy.tail(steps).round(2).to_string())
        yoy.tail(py * 5).plot(figsize=(14, 4), title=f"{a} YoY % (last 5y + forecast)")
        plt.axvline(x=idx[0], color="b", alpha=0.5, linestyle="--")
        plt.show()